In [ ]:
'''
for simplicity when running and debugging, im just printing static images of solutions
to see how it looks in work, checkout README!
(or create a custom function that takes a solution and prints it step by step)
'''

In [ ]:
import math
import random

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
class Candidate:
    def __init__(self, path):
        self.path = path
        self.front_idx = None
        self.crowding_distance = 0.0
        self.objectives = None
        
    def evaluate(self, current_obstacles):
        if self.objectives is None:
            self.objectives = Evaluate(self.path, current_obstacles)
        return self.objectives

In [ ]:
box_shape = 20
# environment = [[0 for x in range(box_shape)] for i in range(box_shape)]  #it was totally unnecessary
start_position = (0,2)
goal_position = (19, 16)


obstacles = (
  (5,1),(5,2),(5,3),(5,4),(5,6),(5,7),(5,8),(5,9),
  (5,11),(5,12),(5,13),(5,14),(5,16),(5,17),(5,18),

  (10,0),(10,1),(10,2),(10,4),(10,5),(10,6),
  (10,8),(10,9),(10,10),(10,12),(10,13),(10,14),
  (10,16),(10,17),(10,18),

  (2,6),(3,6),(4,6),
  (6,10),(7,10),(8,10),
  (11,4),(12,4),(13,4),
  (14,12),(15,12),(16,12),

  (3,14),(4,14),
  (12,8),(13,8),
  (7,16),(8,16)
)

current_obstacles = obstacles
obstacles_change_interval = 50



moves = ("UP", "DOWN", "LEFT", "RIGHT")

num_candidates = 300
upper_limit = int((box_shape**2)/8)
lower_limit = int(box_shape*1)


candidates = []
for i in range(num_candidates):
    length = random.randint(lower_limit, upper_limit)
    path = []
    for j in range(length):
        path.append(random.choice(moves))
    c = Candidate(path)
    candidates.append(c)

In [ ]:
def generate_new_obstacles():
    num_obstacles = random.randint(25, 45)
    new_obs = []
    
    while len(new_obs) < num_obstacles:
        x = random.randint(0, box_shape-1)
        y = random.randint(0, box_shape-1)
        position = (x, y)
        
        if position == start_position or position == goal_position:
            continue
        
        if position not in new_obs:
            new_obs.append(position)
    
    return tuple(new_obs)

In [ ]:
def Evaluate(candidate, current_obstacles):
    collision_count = 0
    x, y = start_position[0], start_position[1]
    new_x, new_y = start_position[0], start_position[1]
    for i in candidate:
        new_x, new_y = x, y
        if i   == "UP": new_y = min(y+1, box_shape-1)
        elif i == "DOWN": new_y = max(y-1, 0)
        elif i == "LEFT": new_x = max(x-1, 0)
        elif i == "RIGHT": new_x = min(x+1, box_shape-1)

        if (new_x,new_y) not in current_obstacles:
            x, y = new_x, new_y
        else:
            collision_count += 1

    distance = abs(goal_position[0] - x) + abs(goal_position[1] - y)
    collision_count = collision_count + distance**3
    length = len(candidate)**0.01 + 1 * distance**5

    objectives = (distance, collision_count, length)
    return objectives

In [ ]:
#trust me i tried all combinations, this works, i didnt make a mistake
def pareto_dominance_calculate(candidate1, candidate2):
    candidate1_values = candidate1.evaluate(current_obstacles)
    candidate2_values = candidate2.evaluate(current_obstacles)
    if candidate1_values == candidate2_values:
        return False
    for i in range(3):
        a = candidate1_values[i] <= candidate2_values[i]
        if a == False:
            return a
    return True

In [ ]:
def pareto_fronts(candidates):
    remaining = candidates.copy()
    fronts = []
    front_count = 0
    while remaining:
        current_front = []
        dominated = []
        for candidate in remaining:
            is_dominated = False
            for other in remaining:
                if candidate is other:
                    continue
                is_dominated = pareto_dominance_calculate(other, candidate)
                if is_dominated:
                    break
            if is_dominated:
                dominated.append(candidate)
            else:
                candidate.front_idx = front_count
                current_front.append(candidate)
        fronts.append(current_front)
        remaining = dominated
        front_count += 1
    return fronts

In [ ]:
def crowding_distance(front):
    if len(front) <= 2:
        for candidate in front:
            candidate.crowding_distance = math.inf
        return

    for candidate in front:
        candidate.crowding_distance = 0.0
        
    num_objectives = 3
    for obj in range(num_objectives):
        front.sort(key = lambda x: x.evaluate(current_obstacles)[obj])
        front[0].crowding_distance = math.inf
        front[-1].crowding_distance = math.inf

        max_value = front[-1].evaluate(current_obstacles)[obj]
        min_value = front[0].evaluate(current_obstacles)[obj]
        obj_range = max_value - min_value
        if obj_range == 0:
            continue
        for i in range(1,len(front)-1):
            front[i].crowding_distance += (front[i+1].evaluate(current_obstacles)[obj]-front[i-1].evaluate(current_obstacles)[obj])/obj_range

In [ ]:
def parent_selection(candidates):
    mating_pool = []
    while len(mating_pool)<num_candidates:
        player_1, player_2 = random.sample(candidates, 2)
        winner = None
        if player_1.front_idx<player_2.front_idx:
            winner = player_1
        elif player_1.front_idx>player_2.front_idx:
            winner = player_2
        elif player_1.crowding_distance>player_2.crowding_distance:
            winner = player_1
        else:
            winner = player_2
        mating_pool.append(winner)
    return mating_pool

def next_gen_selection(parents, offspring):
    combined = parents + offspring
    fronts = pareto_fronts(combined)
    f0 = fronts[0]
    best_dist = min(f0,key = lambda x: x.evaluate(current_obstacles)[0])
    best_coll = min(f0,key = lambda x: x.evaluate(current_obstacles)[1])
    best_len = min(f0,key = lambda x: x.evaluate(current_obstacles)[2])
    bests = [best_dist, best_coll, best_len]
    next_gen = []
    for front in fronts:
        cd = crowding_distance(front)
        if len(next_gen)+len(front) <= num_candidates:
            next_gen.extend(front)
        else:
            front.sort(key = lambda x: x.crowding_distance, reverse=True)
            remaining = num_candidates - len(next_gen)
            next_gen.extend(front[:remaining])
            break
    return next_gen, bests, len(f0)

def crossover(selected):
    next_gen = []
    while len(next_gen)<num_candidates:
        parent_1, parent_2 = random.sample(selected, 2)
        parent_1_path, parent_2_path = parent_1.path, parent_2.path
        if len(parent_1_path)>len(parent_2_path):
            parent_1_path, parent_2_path = parent_2_path, parent_1_path
        slice_idx = random.randint(int(len(parent_1_path)*0.2), int(len(parent_1_path)*0.8))
        child = Candidate(parent_1_path[:slice_idx] + parent_2_path[slice_idx:])
        next_gen.append(child)
    return next_gen


def move_mutation(x):
    for i, j in enumerate(x):
        if random.random()<0.001:
            x[i] = random.choice(moves)
    return x
def insert_delete_mutation(x):
    if random.random()<0.01:
        if random.random()>0.5:
            a = random.choice(moves)
            idx = random.randint(0,len(x)-1)
            x.insert(idx, a)
        else:
            idx = random.randint(0,len(x)-1)
            del x[idx]
    return x

def mutation(next_gen):
    for i in range(len(next_gen)):
        path = next_gen[i].path[:]
        if random.random()>0.5:
            path = move_mutation(next_gen[i].path[:])
        else:
            path = insert_delete_mutation(next_gen[i].path[:])
        next_gen[i] = Candidate(path)
    return next_gen

In [ ]:
'''
    unlike cells above/below, this block is totally AI generated code,
    i read it and tried in many situations and it worked so i didnt change it,
    spending more time with the actual algorithm
'''


def plot_best_path(path, start, goal, obstacles=None, shape=20):
    x_coords = [start[0]]
    y_coords = [start[1]]
    
    curr_x, curr_y = start
    for move in path:
        next_x, next_y = curr_x, curr_y

        if move == "UP": next_y += 1
        elif move == "DOWN": next_y -= 1
        elif move == "LEFT": next_x -= 1
        elif move == "RIGHT": next_x += 1

        next_x = max(0, min(next_x, shape - 1))
        next_y = max(0, min(next_y, shape - 1))

        if obstacles and (next_x, next_y) in obstacles:
            # blocked, stay in place
            pass
        else:
            curr_x, curr_y = next_x, next_y
        
        x_coords.append(curr_x)
        y_coords.append(curr_y)

    # Capture the final position
    final_pos = (x_coords[-1], y_coords[-1])

    plt.figure(figsize=(8, 8))
    
    # Plot Obstacles
    if obstacles:
        obs_x = [o[0] for o in obstacles]
        obs_y = [o[1] for o in obstacles]
        plt.scatter(obs_x, obs_y, color="black", marker="s", s=150, label="Obstacles")

    # Plot Path, Start, and Goal
    plt.plot(x_coords, y_coords, '-', label="Path", color="blue", linewidth=2, alpha=0.8)
    plt.scatter(*start, color="green", s=150, label="Start", zorder=5)
    plt.scatter(*goal, color="red", s=150, label="Goal", marker="X", zorder=5)
    
    # --- New Section: Final Location Text Box ---
    status = "REACHED" if final_pos == goal else "FAILED"
    info_text = f"Final Pos: {final_pos}\nGoal Pos: {goal}\nStatus: {status}"
    
    # transform=plt.gca().transAxes anchors the text to the plot window (0,0 is bottom left, 1,1 is top right)
    plt.text(0.02, 0.98, info_text, transform=plt.gca().transAxes, 
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    # --------------------------------------------

    ticks = np.arange(0, shape + 1, 2) 
    plt.xticks(ticks)
    plt.yticks(ticks)
    plt.xlim(-1, shape)
    plt.ylim(-1, shape)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(loc='upper right')
    plt.title("Pathfinding Visualization")
    plt.show()

In [ ]:
fronts = pareto_fronts(candidates)
for front in fronts:
    crowding_distance(front)

num_gen = 201
best_dist_each_gen = []
best_coll_each_gen = []
best_len_each_gen = []

f0_len_hist = []

immigration_rate = 0.3
num_immigrants = int(len(candidates) * immigration_rate)

for i in range(1, num_gen):

    if i>1 and i % obstacles_change_interval == 0:
        current_obstacles = generate_new_obstacles()
        print(f"GEN {i}: Obstacles changed!")
        for c in candidates:
            c.objectives = None
        candidates = candidates[:-num_immigrants]
        for m in range(num_immigrants):
            length = random.randint(lower_limit, upper_limit)
            path = []
            for n in range(length):
                path.append(random.choice(moves))
            candidates.append(Candidate(path))
            fronts = pareto_fronts(candidates)
            for front in fronts:
                crowding_distance(front)

    parents = parent_selection(candidates)
    offspring = crossover(parents)
    offspring = mutation(offspring)
    candidates, bests, f0_len = next_gen_selection(parents, offspring)

    f0_len_hist.append(f0_len)
    best_dist_each_gen.append(bests[0])
    best_coll_each_gen.append(bests[1])
    best_len_each_gen.append(bests[2])

    if i % 10 == 0:
        print(
        f"gen {i} || "
        f"dist {bests[0].objectives[0]} | "
        f"coll {bests[1].objectives[1]} | "
        f"len {bests[2].objectives[2]} |"
        f"f0_size {f0_len_hist[i-1]}")

    should_plot = (i % 25 == 0 or 
               i % obstacles_change_interval == 0 or
               (best_dist_each_gen[-1] == 0 and best_dist_each_gen[-2] > 0))

    if should_plot:
        dist_path = best_dist_each_gen[-1].path
        plot_best_path(dist_path, start_position, goal_position, current_obstacles, box_shape)

        ##commented out below lines because they end up being same path most of the time
        # coll_path = best_coll_each_gen[-1].path
        # plot_best_path(coll_path, start_position, goal_position, obstacles, box_shape)

        # len_path = best_len_each_gen[-1].path
        # plot_best_path(len_path, start_position, goal_position, obstacles, box_shape)